# Summit 2026 HOL — CoWork
## Module 04: Deploy the Agent for Business Users

_Icons used throughout this lab: 🛠️ Action  📌 Note  🔹 Info_

---

### What This Module Does:

1. 🛠️ Deploys the Cortex Agent you built in Module 03 as a **CoWork** experience
2. 🛠️ Configures the SI chat interface for business users
3. 🛠️ Shows how to **share** the agent with non-technical users
4. 🛠️ Creates an **SI-only user** to demonstrate scoped access
5. 🔹 Compares the developer (API) vs. business user (SI) experience side by side

---

### What is CoWork?

**CoWork** is the governed, self-service chat UI built into Snowsight. It lets business users — portfolio managers, compliance officers, executives — ask questions in natural language without writing SQL, calling APIs, or understanding data models.

Key differences from the Cortex Agent API:

| Aspect | Cortex Agent (Module 03) | CoWork (this module) |
|--------|--------------------------|--------------------------------------|
| Who uses it | Developers embed it in apps | Business users open it in Snowsight |
| How to access | SQL function or REST API | Chat UI (elephant icon in Snowsight) |
| SQL visibility | Caller sees generated SQL | Business user sees answers only (SQL hidden) |
| Sharing | Grant USAGE on agent object | Share via link or assign SI-only user access |
| MCP connectors | Not applicable (API-only) | Users can connect to Jira, Salesforce, Slack |
| Conversation | Managed via thread IDs in code | Built-in chat history in UI |

**Same agent. Same semantic view. Same tools. Different delivery for different audiences.**

> **Documentation:** [CoWork (formerly Snowflake Intelligence)](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-intelligence)

---

### Want the Full SI Deep-Dive?

This module covers SI fundamentals within the Summit HOL flow. For **comprehensive L200 proficiency** — including SI-only users, semantic view enhancements, search services, agent tool extensions, custom branding, cost dashboards, and CoCo customization — complete the full hands-on lab:

**[CoWork GA Hands-On Lab (~4 hours, 13 modules)](https://github.com/snowflake-corp/collegeai/blob/initial-import/SI_GA_HOL/README.md)**

> This Summit 2026 HOL only covers concepts from Modules 04 and 08 of the CoWork GA HOL. The CoWork GA HOL goes significantly deeper on pricing, search services, agent tool extension, custom branding, and demo customization with CoCo.

---

### Estimated Time: **20–25 minutes**

### Prerequisites:
- Module 03 complete (NEXUS_AGENT created)

## Step 1: Access CoWork (formerly Snowflake Intelligence) in Snowsight

### How to Find SI:

1. In Snowsight, look at the **left navigation** panel
2. Click **AI & ML → Cortex AI → Agents**
3. You should see `NEXUS_AGENT` listed (created in Module 03)
4. Click on it to open the agent management page

Alternatively, you can access agents that have been shared with you via the **CoWork** entry point (the chat icon).

---

### What You'll See:

The Agent page shows:
- **Configuration** — The spec you defined (model, tools, instructions)
- **Test** — A chat interface to test the agent
- **Threads** — Conversation history
- **Monitoring** — Usage, latency, feedback
- **MCP Connectors** — External tools (we'll set these up in Module 05)

> **Try it now:** Click into the agent and use the "Test" tab to ask: *"Who are our top clients?"* — you should get the same answer as Module 03, but in a chat UI.

## Step 2: 🛠️ Create an SI-Only Role

### Give Access Without Giving SQL

A powerful governance feature of CoWork is **SI-only users**. These are Snowflake users who:
- Can chat with agents via the SI UI
- Cannot write or see SQL
- Cannot access worksheets or notebooks
- Cannot see raw table data
- Only see the natural language answers the agent provides

This is how you give a VP of Sales or a compliance officer access to data insights without granting them database access.

The key is `ALLOWED_INTERFACES = ('SNOWFLAKE_INTELLIGENCE')` — this restricts the user to ONLY the CoWork chat
experience. They interact with data exclusively through the agent.

> **Documentation:** [User access and settings for 
agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-intelligence/deploy-agents)
>
> **For a deeper dive on SI-only users** (SCIM provisioning, SSO, multiple agents): See the [CoWork GA HOL Module 05](https://github.com/snowflake-corp/collegeai/blob/initial-import/SI_GA_HOL/05_SI_Only_Users/README.md)

📌 In Snowflake Demo accounts, SI-only user isolation may not behave exactly like production because demo users often retain inherited demo/account privileges. The intended production pattern is ALLOWED_INTERFACES = ('SNOWFLAKE_INTELLIGENCE'), but this lab demonstrates the role/grant pattern rather than proving full UI isolation.

In [ ]:
%%sql -r SIOnlyRoleConfig
USE ROLE ACCOUNTADMIN;

-- Create a role for SI-only access
CREATE ROLE IF NOT EXISTS NEXUS_SI_USER
  COMMENT = 'Role for business users who access Nexus Capital agent via CoWork only';

-- Grant minimal privileges: just enough to use the agent via SI
GRANT USAGE ON DATABASE NEXUS_HOL TO ROLE NEXUS_SI_USER;
GRANT USAGE ON SCHEMA NEXUS_HOL.AGENTS TO ROLE NEXUS_SI_USER;
GRANT USAGE ON AGENT NEXUS_HOL.AGENTS.NEXUS_AGENT TO ROLE NEXUS_SI_USER;
GRANT USAGE ON WAREHOUSE NEXUS_WH TO ROLE NEXUS_SI_USER;

-- Grant Cortex Agent access (required to call the agent)
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE NEXUS_SI_USER;

-- Grant the SI role to your user so you can test it via role switch
GRANT ROLE NEXUS_SI_USER TO ROLE ACCOUNTADMIN;

SELECT 'NEXUS_SI_USER role created with scoped agent access' AS STATUS;

### What NEXUS_SI_USER Can Do:
- Chat with NEXUS_AGENT via CoWork
- See natural language answers
- Use MCP connectors (Module 05)

### What NEXUS_SI_USER Cannot Do:
- Run SQL queries directly
- Access raw tables
- See the generated SQL behind answers
- Modify the agent configuration
- Access other schemas or databases

This is **governed AI access** — the user gets insights without exposure to underlying data structures.

## Step 3: 🛠️ Test as the SI-Only Role

### Simulate the business user experience

Switch to the `NEXUS_SI_USER` role in your current Snowsight session:

1. Click your name (bottom left) → **Switch Role** → `NEXUS_SI_USER`
2. Navigate to **AI & ML → Cortex AI → Agents** in the left navigation
3. Select `NEXUS_AGENT`/**Nexus Capital Analyst** if you customized the display name in Step 5. Try these questions:
   - "Who are our top clients by AUM?"
   - "Are there any compliance concerns?"
   - "Show me technology sector exposure as a chart"

> **Note:** You're using the Agent Admin "Test" tab here — this is the builder's testing interface. In Step 4, you'll see the
  CoWork chat, which is the clean, no-admin experience that business users see.
  
### What to notice:

- The agent returns the same answers regardless of which role calls it
- Business users interact in natural language — no SQL knowledge required
- Suggested follow-up questions guide the conversation

> **In production:** You'd pair this role with `ALLOWED_INTERFACES = ('SNOWFLAKE_INTELLIGENCE')` on the user object, restricting them to ONLY the SI chat at `ai.snowflake.com` — no worksheets, no data browser, no SQL. For the full walkthrough (including SSO-based MFA), see the [CoWork GA HOL Module 05](https://github.com/snowflake-corp/collegeai/blob/initial-import/SI_GA_HOL/05_SI_Only_Users/README.md).

 ### 🧠 Going Deeper: Agent Monitoring, Traces & Evaluations

Every question you just asked generated a full trace — tool selection, SQL generated, search results retrieved, reasoning steps.
As an agent owner, you can use this to understand and improve quality.

| Capability | What It Shows | Where to Find It |
|-----------|---------------|-----------------|
| **Traces** | Full execution path for each request (tools called, SQL run, results returned) | AI & ML → Cortex AI → Agents → NEXUS_AGENT → Monitoring |
| **Evaluations** | Measure agent accuracy against a test set of questions with expected answers | CREATE AGENT EVALUATION (SQL) |
| **Feedback** | End-user thumbs up/down on answers — feeds into refinement loop | Built into SI chat UI automatically |

---

**Not required for this lab** — but if you're deploying SI to production or running customer workshops, observability is how you prove accuracy and continuously improve. Please check out these resources for more information:

| Resource | Link |
|----------|------|
| AI Observability Resources | [GDoc](https://docs.google.com/document/d/1DOt78YPC4h-8FI0WNFZLCeaopyIxAOH2Nm3yAMGeSz0/edit?usp=sharing)|
| AI Observability customer-facing PPT | [GSlides](https://docs.google.com/document/d/1DOt78YPC4h-8FI0WNFZLCeaopyIxAOH2Nm3yAMGeSz0/edit?usp=sharing) |
| Monitor Cortex Agent Requests | [Documentation](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents#monitor-evaluate-and-iterate) |
| Cortex Agent Evaluations | [Documentation](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-evaluations) |
| Best Practices for Evaluating Agents | [Documentation](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-evaluations) |

> **Try it now (optional):** Go to AI & ML → Cortex AI → Agents → NEXUS_AGENT → Monitoring. Click on any recent request to see the full trace — which tools were called, what SQL was generated, and how the agent reasoned through the answer.

### 🔄 The Improvement Flywheel

As you test the agent and observe results, the refinement loop is:

1. **Ask questions** → review answers
2. **Wrong answer?** → Add a verified query to the semantic view (Module 02 pattern)
3. **Wrong tool selected?** → Refine orchestration instructions in the agent spec
4. **Missing context?** → Update tool descriptions or add SQL generation instructions

This is continuous — the best agents get refined over weeks based on real usage patterns. You already have the building blocks: verified queries (Module 02), agent instructions (Module 03), and observability (above).

## Step 4: 🛠️ Explore the SI Chat Experience

### Interact with the agent as a business user would:

> 📌 **Important:** If clicking CoWork (formerly Snowflake Intelligence) in the left nav redirects you to a different account (e.g., Snowhouse), use an
Incognito window to access SI on your demo account directly.

**Steps:**

1. Open an **Incognito/Private window** (Chrome: Cmd+Shift+N | Windows: Ctrl+Shift+N)
2. Navigate to `https://app.snowflake.com` and log into your **demo account**
3. Open CoWork/Snowflake Intelligence from the left navigation. If it is not visible, open the Agent page for NEXUS_AGENT and use `+ Add Agent` to make it available in CoWork.
4. Select **Nexus Capital Analyst** from the agent list

### Before you start chatting — notice the "+" menu:

Click the **+** icon in the chat input area. This reveals:

| Option | What It Does |
|--------|-------------|
| **@ Upload files** | Attach CSV, Excel, or other files for the agent to analyze |
| **# Skills** | Access specialized agent skills |
| **@ Connectors** | Connect to external tools (Jira, Salesforce — we'll do this in Module 05) |

> **Extended thinking:** You'll see the agent's reasoning steps as it works. This is useful for understanding *how* it arrived at
an answer. For complex questions, the agent takes longer but produces more accurate, multi-step responses. The orchestration budget
in the agent spec (`budget.seconds` and `budget.tokens`) controls how much reasoning the agent can do per request.

### Try these conversations:

---

**Conversation 1: Portfolio Overview**
- "Give me a summary of our portfolio performance"
- "Which sectors are we most concentrated in?"
- "Show me that as a chart" ← (triggers data_to_chart tool)

---

**Conversation 2: Client Deep-Dive**
- "Tell me about Velocity Capital Partners"
- "What positions do they hold?"
- "Are there any research notes about their holdings?"

---

**Conversation 3: Compliance Check**
- "Are there any compliance flags I should know about?"
- "Which clients have single-stock concentration above 10%?"

### How the SI Chat Differs from the Agent Admin "Test" Tab

Now that you've used both, here's what's different:

| | Agent Admin → Test | CoWork Chat |
|---|---|---|
| Full agent spec visible | Yes — edit tools, instructions, resources | No |
| Monitoring/traces accessible | Same page | Separate (admin only) |
| Who sees this | Agent builders | Business users |
| MCP Connectors | Configure (add/remove) | Connect & use (per-user OAuth) |
| Save/Share/Explore responses | No | Yes — save answers, share with colleagues, explore data further |
| Upload files for ad-hoc analysis | No | Yes (via + menu) |

### Both experiences share:
- Thinking/reasoning steps visible (expandable)
- Generated SQL visible (expandable)
- Multi-turn conversation context
- Inline visualizations and charts
- Cited data sources
- Suggested follow-up questions

### 🔹 Artifacts: Save and Share Insights

After the agent generates charts or tables, you can **save them as Artifacts** — persistent, shareable snapshots of key analyses.

**How it works:**
- Click **Save** on a chart or table in a response
- Saved items appear in the **Artifacts** panel (left sidebar)
- Share artifacts with colleagues via link — they can revisit the saved result without re-asking the question
- Great for recurring analyses: save once, reference anytime

> **Try it:** After running Conversation 1, save the sector concentration chart as an artifact. Then open the Artifacts panel to confirm it persisted.

> **Note:** Available artifact actions and UI behavior may vary slightly depending on the current Snowsight / Snowflake Intelligence release.

**RBAC still applies:**  
Sharing an artifact shares the *saved visual output* (chart or table), but if a colleague clicks **Explore** or re-runs the underlying query, their role permissions determine what data they can access. A shared artifact does **not** bypass Snowflake access controls — it is a snapshot, not a privilege escalation.

---

### 🔹 PDF Export *(Coming Soon)*

PDF export for Snowflake Intelligence responses is expected in a future PuPr release. This will allow users to export agent responses as formatted PDFs for sharing with stakeholders who may not have direct Snowflake access.

## Step 5: 🛠️ Customize the Agent Experience

### Make the agent feel like a branded internal tool

Navigate to **AI & ML → Cortex AI → Agents → `NEXUS_AGENT` → Edit → About** and click the icon in the top right to customize:

| Option | What It Does |
|--------|-------------|
| **Color** | Pick from 7 color options (blue, purple, green, red, etc.) |
| **Icon** | Choose from a pre-defined icon set (chart, globe, shield, lightbulb, etc.) |
| **Display name** | What users see in the agent list |
| **Description** | Appears below the name — helps users understand what to ask |
| **Example questions** | Conversation starters shown when users first open the agent |

> **Try it:** Change the color and icon to something that fits "financial analyst" — then check how it looks in CoWork chat.

You can also set display name and color via SQL:

```sql
ALTER AGENT NEXUS_HOL.AGENTS.NEXUS_AGENT SET
  COMMENT = 'Nexus Capital AI Analyst - Ask questions about clients, portfolios, trades, and market research',
  PROFILE = '{"display_name": "Nexus Capital Analyst", "color": "red"}';
```

Why this matters: When deploying to 50+ business users, the agent should feel like their tool. Naming it "Revenue Analyst" or
"Supply Chain Assistant" (not "PROD_AGENT_V2"), picking a recognizable icon, and adding sample questions reduces time-to-first-value
and builds trust.

📌 Color/icon changes currently appear in the Agent view. They may not immediately appear in CoWork/SI, depending on the current UI release/cache behavior.

In [ ]:
%%sql -r dataframe_2
-- Update the agent profile (display name, description, color)
ALTER AGENT NEXUS_HOL.AGENTS.NEXUS_AGENT SET
    COMMENT = 'Nexus Capital AI Analyst - Ask questions about clients, portfolios, trades, and market research',
    PROFILE = '{
    "display_name": "Nexus Capital Analyst",
    "avatar": "shield",
    "color": "red"
    }';

## Step 6: 🛠️ Track SI Usage

### Account Usage views for AI consumption

Beyond the traces and evaluations covered earlier, these views help you track SI adoption and costs.

The following query shows recent agent usage — who called it, when, token consumption, and credits:

In [ ]:
%%sql -r TrackSIUsage
-- Check if there are any agent usage records
-- (This view captures usage after interactions occur)
SELECT START_TIME, END_TIME, USER_NAME, AGENT_NAME, TOKEN_CREDITS, TOKENS
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
    WHERE AGENT_NAME = 'NEXUS_AGENT'
    ORDER BY START_TIME DESC
    LIMIT 10;

> **What to tell customers:** AI usage is fully visible in Account Usage views — same place you already monitor warehouse and
storage costs. No surprises, no hidden meters.
>
> **Documentation:** [Cortex Agent Usage 
History](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_agent_usage_history) | [Metering Daily 
History](https://docs.snowflake.com/en/sql-reference/account-usage/metering_daily_history)

## ✅ Module 04 Complete!

### You now have:
- The same Cortex Agent accessible via **two channels**:
  - **Developer API** (Module 03) — for embedding in applications
  - **CoWork UI** (this module) — for business user self-service
- An **SI-only user role** (`NEXUS_SI_USER`) demonstrating governed access
- Agent display configured for the CoWork chat UI

---

### Key Takeaways:

1. **One agent, two delivery models.** Build once with Cortex Agent, deliver via API for developers AND via SI for business users.

2. **Governance is built in.** SI-only users get answers without SQL access. The agent enforces RBAC — users only see data their role permits.

3. **The semantic view is the foundation.** Both the API and SI rely on the same semantic view for accurate SQL generation. Invest in your semantic layer = better answers everywhere.

---

### Talking Points:

- "CoWork is not a separate product — it's a governed delivery channel for the Cortex Agents you already built."
- "SI-only users get data access through AI, not through SQL. That's a fundamentally different access pattern."
- "The same verified queries that power the API also power SI — one source of truth."

---

### Next Steps:

Continue to **Module 05: MCP Connectors** — Connect the SI agent to external enterprise systems (Jira, Salesforce, Slack) so it can take action, not just answer questions.